# 02 · Export HLS chips

For every fire in `fires.csv` (plus the Wapiti Fire from its RECOVER perimeter), this notebook:

1. Cuts the area around the fire into 224 × 224 chips at 30 m, the input size Prithvi expects.
2. Builds one image per chip: a few static layers plus a **monthly HLS composite** (6 bands + number of clear looks) from 2 years before the fire until last month.
3. Exports each chip as a GeoTIFF to Google Drive, plus one CSV with monthly weather (gridMET) for every chip.

Run it top to bottom. With `MODE = "test"` it only exports one chip, so we can check the file before sending everything.
Exports run on Earth Engine's servers, so you can close the laptop while they run. Rerun section 6 to see how they're doing.

## 0. Settings

In [ ]:
import os

# Earth Engine project ID, same as notebook 01 (ee_project.txt at the repo root, git-ignored)
PROJECT_ID = ""
if not PROJECT_ID:
    for path in ("ee_project.txt", "../ee_project.txt"):
        if os.path.exists(path):
            PROJECT_ID = open(path).read().strip()
assert PROJECT_ID, "Put your Earth Engine project ID in ee_project.txt at the repo root"

# Run from notebooks/ (VS Code). In Colab, clone the repo and point REPO to it.
REPO = ".."
FIRES_CSV = f"{REPO}/data/fires/fires.csv"
WAPITI_GEOJSON = f"{REPO}/data/fires/wapiti_perimeter.geojson"
MANIFEST_CSV = f"{REPO}/data/chips_manifest.csv"
EXPORT_LOG = f"{REPO}/data/raw/exports_log.csv"   # git-ignored, just for us
FIG_DIR = f"{REPO}/figures"

DRIVE_FOLDER = "cs497_hls_chips"   # Drive folder the exports go to

CHIP = 224                # pixels per side, what Prithvi expects
RES = 30                  # HLS pixel size in meters
MIN_INSIDE = 0.3          # keep a chip if at least 30% of it is inside the perimeter
MAX_CHIPS_PER_FIRE = 25   # so the biggest fires don't take over the dataset
YEARS_BEFORE = 2          # years of images before the fire (pre-fire baseline)
END_MONTH = None          # last month to export, e.g. "2026-10"; None = last full month
MONTHS_PER_FILE = None    # None = one file per chip; set e.g. 48 if exports fail for memory
SEED = 497
REBUILD_MANIFEST = False  # True = recompute the chip grid (changes the chips!)

MODE = "test"             # "test" = export 1 chip, "all" = every chip, "none" = just look
ROWS = None               # only export manifest rows [start, end), e.g. (148, 225), to split the work
                          # between Earth Engine accounts (each account has its own queue)

In [ ]:
import json
import math
import re

import numpy as np
import pandas as pd
import geopandas as gpd
import shapely.geometry
import matplotlib.pyplot as plt
import ee
from IPython.display import Image, display

ee.Authenticate()
ee.Initialize(project=PROJECT_ID)
for d in (os.path.dirname(EXPORT_LOG), FIG_DIR):
    os.makedirs(d, exist_ok=True)

if END_MONTH is None:
    END_MONTH = str(pd.Timestamp.today().to_period("M") - 1)
print("Exporting through", END_MONTH)

## 1. Fires and perimeters

The study fires come from `fires.csv` (step 1) and their perimeters from MTBS.
Wapiti isn't in MTBS yet, so we use the final perimeter from its RECOVER package (Oct 19, 2024), simplified to 15 m so it's small enough to send to Earth Engine.

In [ ]:
MTBS = ee.FeatureCollection("USFS/GTAC/MTBS/burned_area_boundaries/v1")
WAPITI_ID = "WAPITI2024"

fires = pd.read_csv(FIRES_CSV)

wapiti_geom = None
if os.path.exists(WAPITI_GEOJSON):
    shape = gpd.read_file(WAPITI_GEOJSON)
    utm = shape.estimate_utm_crs()
    simple = shape.to_crs(utm).simplify(15)
    geojson = json.loads(json.dumps(simple.to_crs(4326).iloc[0].__geo_interface__))  # tuples -> lists
    wapiti_geom = ee.Geometry(geojson, None, False)
    if not fires["name"].str.upper().eq("WAPITI").any():
        c = simple.centroid.to_crs(4326).iloc[0]
        fires = pd.concat([fires, pd.DataFrame([{
            "name": "WAPITI", "mtbs_id": WAPITI_ID, "year": 2024, "state": "ID", "region": "Rockies",
            "acres": shape["acres"].iloc[0], "lat": c.y, "lon": c.x, "role": "case_study",
        }])], ignore_index=True)


def fire_geometry(row):
    if row["mtbs_id"] == WAPITI_ID:
        return wapiti_geom
    return ee.Feature(MTBS.filter(ee.Filter.eq("Event_ID", row["mtbs_id"])).first()).geometry()


def perimeter_mask(perim):
    """1 inside the perimeter, 0 outside."""
    return ee.Image(0).paint(ee.FeatureCollection([ee.Feature(perim)]), 1)


fires[["name", "year", "state", "region", "veg_type", "severity", "acres", "role"]]

## 2. Chip grid

For each fire we lay a grid of 224 × 224 chips (6.72 km) over its bounding box, in the fire's UTM zone and lined up with the 30 m HLS pixels.
We keep chips that are at least 30% inside the perimeter (the rest of the chip is unburned land nearby, which is useful as a reference), and at most 25 per fire.

The grid is saved in `data/chips_manifest.csv`, so the chips stay the same every time we run this.

In [ ]:
SIZE = CHIP * RES  # chip side in meters


def utm_epsg(lon, lat):
    zone = int((lon + 180) // 6) + 1
    return f"EPSG:{(32600 if lat >= 0 else 32700) + zone}"


def tile_grid(xmin, ymin, xmax, ymax):
    """Upper-left corners of the chips that cover a box, snapped to the 30 m grid."""
    x0 = math.floor(xmin / RES) * RES
    y0 = math.ceil(ymax / RES) * RES
    nx = math.ceil((xmax - x0) / SIZE)
    ny = math.ceil((y0 - ymin) / SIZE)
    return [(x0 + i * SIZE, y0 - j * SIZE) for j in range(ny) for i in range(nx)]


def slug(name):
    return re.sub(r"[^A-Z0-9]+", "", str(name).upper())[:16]


def chip_region(epsg, x, y):
    x, y = float(x), float(y)  # numpy ints don't serialize for Earth Engine
    return ee.Geometry.Rectangle([x, y - SIZE, x + SIZE, y], ee.Projection(epsg), False)


def pick_chips(corners, shares, key):
    df = pd.DataFrame([{"x": x, "y": y, "inside": shares.get(k, 0.0)} for k, (x, y) in enumerate(corners)])
    df = df[df["inside"] >= MIN_INSIDE]
    if len(df) > MAX_CHIPS_PER_FIRE:
        df = df.sample(MAX_CHIPS_PER_FIRE, random_state=SEED)
    df = df.sort_values(["y", "x"], ascending=[False, True]).reset_index(drop=True)
    df["chip_id"] = [f"{key}_{i:02d}" for i in range(len(df))]
    return df


def chips_for_fire(row):
    epsg = utm_epsg(row["lon"], row["lat"])
    perim = fire_geometry(row)
    ring = perim.bounds(RES, ee.Projection(epsg)).coordinates().get(0).getInfo()
    xs, ys = zip(*ring)
    corners = tile_grid(min(xs), min(ys), max(xs), max(ys))
    tiles = ee.FeatureCollection([ee.Feature(chip_region(epsg, x, y), {"k": k})
                                  for k, (x, y) in enumerate(corners)])
    inside = perimeter_mask(perim).rename("inside")
    stats = inside.reduceRegions(tiles, ee.Reducer.mean(), scale=90, crs=epsg).getInfo()
    shares = {f["properties"]["k"]: f["properties"].get("mean") or 0.0 for f in stats["features"]}
    key = f"{row['state']}{int(row['year'])}_{slug(row['name'])}"
    df = pick_chips(corners, shares, key)
    for col in ("mtbs_id", "name", "year", "region", "veg_type", "severity", "role"):
        df[col] = row.get(col)
    df["epsg"] = epsg
    return df


def months_for(year):
    return [str(p) for p in pd.period_range(f"{int(year) - YEARS_BEFORE}-01", END_MONTH, freq="M")]

In [ ]:
MANIFEST_COLUMNS = ["chip_id", "mtbs_id", "name", "year", "region", "veg_type", "severity", "role",
                    "epsg", "x", "y", "inside", "start_month", "end_month", "n_months"]

# Chips already in the manifest never change; only fires that are new in fires.csv get chips
if os.path.exists(MANIFEST_CSV) and not REBUILD_MANIFEST:
    manifest = pd.read_csv(MANIFEST_CSV)
    print("Loaded", MANIFEST_CSV, f"({len(manifest)} chips)")
else:
    manifest = pd.DataFrame(columns=MANIFEST_COLUMNS)

new_fires = fires[~fires["mtbs_id"].isin(manifest["mtbs_id"])]
if len(new_fires):
    parts = []
    for _, row in new_fires.iterrows():
        df = chips_for_fire(row)
        print(f"{row['name']:<24} {len(df):3d} chips")
        parts.append(df)
    new = pd.concat(parts, ignore_index=True)
    new["start_month"] = new["year"].map(lambda y: months_for(y)[0])
    new["end_month"] = END_MONTH
    new["n_months"] = new["year"].map(lambda y: len(months_for(y)))
    manifest = pd.concat([manifest, new[MANIFEST_COLUMNS]], ignore_index=True)
    manifest.to_csv(MANIFEST_CSV, index=False)
    print(f"Added {len(new)} chips from {len(new_fires)} new fires")

n_bands = 6 + 7 * manifest["n_months"]
size_gb = (n_bands * CHIP * CHIP * 2).sum() / 1e9
print(f"{len(manifest)} chips from {manifest['mtbs_id'].nunique()} fires, about {size_gb:.0f} GB on Drive (before compression)")
manifest.groupby(["name", "year"])["chip_id"].count().sort_values(ascending=False).to_frame("chips").T

Chips over each perimeter (saved to `figures/chips_per_fire.png` for the report).

In [ ]:
def outline(geojson):
    parts = geojson.get("geometries", [geojson])
    for part in parts:
        polygons = {"Polygon": [part["coordinates"]], "MultiPolygon": part["coordinates"]}.get(part["type"], [])
        for polygon in polygons:
            for ring in polygon:
                yield ring


fire_ids = list(dict.fromkeys(manifest["mtbs_id"]))  # names can repeat (two East Fork fires)
ncols = 4
nrows = math.ceil(len(fire_ids) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
for ax, fid in zip(axes.flat, fire_ids):
    chips = manifest[manifest["mtbs_id"] == fid]
    epsg = chips["epsg"].iloc[0]
    row = fires[fires["mtbs_id"] == fid].iloc[0]
    geom = fire_geometry(row).transform("EPSG:4326", 30).simplify(90).getInfo()
    utm_geom = gpd.GeoSeries([shapely.geometry.shape(geom)], crs=4326).to_crs(epsg).iloc[0]
    for ring in outline(shapely.geometry.mapping(utm_geom)):
        ax.plot(*zip(*ring), color="firebrick", linewidth=0.8)
    for _, c in chips.iterrows():
        ax.add_patch(plt.Rectangle((c["x"], c["y"] - SIZE), SIZE, SIZE, fill=False, color="steelblue", linewidth=0.8))
    ax.set_title(f"{row['name'].title()} ({int(row['year'])}) · {len(chips)} chips", fontsize=9)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
for ax in list(axes.flat)[len(fire_ids):]:
    ax.axis("off")
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/chips_per_fire.png", dpi=150)
plt.show()

## 3. What goes in each chip

Every chip is one GeoTIFF (int16) with these bands, in this order:

| Bands | What | Units |
|---|---|---|
| `PERIM` | 1 inside the fire perimeter, 0 outside | – |
| `SEVERITY` | MTBS severity class for the fire year (0 none, 1 unburned/low … 4 high, 5 greener, 6 no data) | class |
| `NLCD` | NLCD land cover from before the fire | class |
| `ELEV`, `SLOPE_X100`, `ASPECT` | SRTM terrain | m, degrees × 100, degrees |
| `YYYY-MM_BLUE` … `YYYY-MM_SWIR2` | median of the clear HLS looks that month, 6 Prithvi bands | reflectance × 10,000 |
| `YYYY-MM_N` | how many clear looks went into that month | count |

Months with no clear look are `-9999`. The months go from January two years before the fire to `END_MONTH`, so every fire has its own length (it's in the manifest).

HLS: Landsat (L30) and Sentinel-2 (S30) are merged after renaming their bands to the same 6 names. Pixels flagged as cloud, next to cloud, cloud shadow, snow/ice or high aerosol in `Fmask` are dropped.

In [ ]:
L30 = ee.ImageCollection("NASA/HLS/HLSL30/v002")
S30 = ee.ImageCollection("NASA/HLS/HLSS30/v002")
BANDS = ["BLUE", "GREEN", "RED", "NIR", "SWIR1", "SWIR2"]
L30_BANDS = ["B2", "B3", "B4", "B5", "B6", "B7"]
S30_BANDS = ["B2", "B3", "B4", "B8A", "B11", "B12"]
STATIC = ["PERIM", "SEVERITY", "NLCD", "ELEV", "SLOPE_X100", "ASPECT"]
NODATA = -9999
SCALE = 10_000   # checked below

# Fully masked image, so a month with no looks still gives 6 (empty) bands
EMPTY = ee.Image.constant([0] * 6).rename(BANDS).toFloat().updateMask(0)


def clear_pixels(img):
    qa = img.select("Fmask")
    bad = (qa.bitwiseAnd(0b11110).neq(0)                # bits 1-4: cloud, next to cloud, shadow, snow/ice
           .Or(qa.rightShift(6).bitwiseAnd(3).eq(3)))   # bits 6-7 = 3: high aerosol
    return img.select(BANDS).toFloat().updateMask(bad.Not())


def hls(region, start, end):
    l30 = L30.filterBounds(region).filterDate(start, end).select(L30_BANDS + ["Fmask"], BANDS + ["Fmask"])
    s30 = S30.filterBounds(region).filterDate(start, end).select(S30_BANDS + ["Fmask"], BANDS + ["Fmask"])
    return l30.merge(s30).map(clear_pixels)


def monthly_stack(region, months):
    col = hls(region, f"{months[0]}-01", ee.Date(f"{months[-1]}-01").advance(1, "month"))
    out = []
    for m in months:
        start = ee.Date(f"{m}-01")
        month = col.filterDate(start, start.advance(1, "month")).merge(ee.ImageCollection([EMPTY]))
        refl = month.median().multiply(SCALE).round().unmask(NODATA)
        n = month.select("BLUE").count().unmask(0).rename("N")
        out.append(ee.Image.cat([refl, n]).rename([f"{m}_{b}" for b in BANDS + ["N"]]))
    return ee.Image.cat(out)

In [ ]:
SEVERITY_COL = ee.ImageCollection("USFS/GTAC/MTBS/annual_burn_severity_mosaics/v1")
NLCD_COL = ee.ImageCollection("USGS/NLCD_RELEASES/2019_REL/NLCD")
NLCD_EPOCH = {2015: 2013, 2016: 2013, 2017: 2016, 2018: 2016, 2019: 2016,
              2020: 2019, 2021: 2019, 2022: 2019, 2023: 2019, 2024: 2019}
DEM = ee.Image("USGS/SRTMGL1_003").select("elevation")
TERRAIN = ee.Terrain.products(DEM)


def image_table(col):
    """id, year and study area of every image in a collection (same as notebook 01)."""
    rows = []
    for im in col.getInfo()["features"]:
        p = im.get("properties", {})
        t = p.get("system:time_start")
        years = re.findall(r"(?:19|20)\d\d", im["id"])
        year = pd.Timestamp(t, unit="ms").year if t is not None else (int(years[-1]) if years else None)
        rows.append({"id": im["id"], "year": year, "area": str(p.get("study_area", "")).upper()})
    return pd.DataFrame(rows)


SEV_TABLE = image_table(SEVERITY_COL)
NLCD_TABLE = image_table(NLCD_COL)


def severity_image(year):
    t = SEV_TABLE[SEV_TABLE["year"] == year]
    if len(t) > 1:
        t = t[(t["area"] == "CONUS") | t["id"].str.upper().str.contains("CONUS")]
    if len(t) != 1:
        print(f"   no MTBS severity mosaic for {year}, SEVERITY will be 0")
        return ee.Image(0)
    return ee.Image(t["id"].iloc[0]).select("Severity")


def nlcd_image(year):
    t = NLCD_TABLE[NLCD_TABLE["year"] == NLCD_EPOCH[int(year)]]
    return ee.Image(t["id"].iloc[0]).select("landcover")


def static_bands(perim, year):
    layers = [
        perimeter_mask(perim),
        severity_image(int(year)).unmask(0),
        nlcd_image(year).unmask(0),
        DEM,
        TERRAIN.select("slope").multiply(100).round(),
        TERRAIN.select("aspect").round(),
    ]
    return ee.Image.cat([img.rename(name) for img, name in zip(layers, STATIC)])


PERIMS = {}


def chip_image(chip, months):
    if chip["mtbs_id"] not in PERIMS:
        PERIMS[chip["mtbs_id"]] = fire_geometry(fires[fires["mtbs_id"] == chip["mtbs_id"]].iloc[0])
    region = chip_region(chip["epsg"], chip["x"], chip["y"])
    img = ee.Image.cat([static_bands(PERIMS[chip["mtbs_id"]], chip["year"]), monthly_stack(region, months)])
    return img.toInt16(), region

## 4. Quick check before exporting

Earth Engine serves HLS as reflectance (0–1); Prithvi was trained on HLS × 10,000, so we multiply by 10,000. This cell checks that, and shows one chip before the fire, one year after and last summer (false color: SWIR2, NIR, red; burned ground looks red/brown, vegetation green).

In [ ]:
test = manifest.sort_values("inside", ascending=False).iloc[0]
test_region = chip_region(test["epsg"], test["x"], test["y"])
y = int(test["year"])

red = (hls(test_region, f"{y - 1}-06-01", f"{y - 1}-09-01").select("RED").median()
       .reduceRegion(ee.Reducer.median(), test_region, 30).get("RED").getInfo())
print(f"Median red reflectance before the fire: {red}")
if red is not None and red > 2:
    SCALE = 1
    print("Values already look scaled by 10,000, so SCALE = 1")

last_summer = f"{int(END_MONTH[:4]) - (1 if int(END_MONTH[5:]) < 8 else 0)}-08"
for label, m in (("before", f"{y - 1}-08"), ("1 year after", f"{y + 1}-08"), ("last summer", last_summer)):
    img = monthly_stack(test_region, [m]).select([f"{m}_SWIR2", f"{m}_NIR", f"{m}_RED"])
    url = img.getThumbURL({"region": test_region, "dimensions": 256, "min": 0, "max": 4000, "crs": test["epsg"]})
    print(f"{test['chip_id']} · {label} ({m})")
    display(Image(url=url, embed=True, format="png"))

## 5. Export the chips

`MODE = "test"` sends one chip; check the file in Drive (`cs497_hls_chips`), then set `MODE = "all"` and rerun this cell.
Chips that are already done or still running are skipped, so it's safe to rerun (it also resends any that failed).

To split the work between Earth Engine accounts, give each one a different `ROWS` range of `data/chips_manifest.csv`. Each account only sees its own tasks, so the ranges must not overlap. Each account exports to its own Drive; move the files into one folder afterwards.

In [ ]:
LOG_COLUMNS = ["file", "chip_id", "task_id", "submitted"]
OK_STATES = ("PENDING", "RUNNING", "SUCCEEDED")


def chip_files(chip):
    """(file name, months) for each GeoTIFF of a chip."""
    months = months_for(chip["year"])
    if not MONTHS_PER_FILE:
        return [(chip["chip_id"], months)]
    blocks = [months[i:i + MONTHS_PER_FILE] for i in range(0, len(months), MONTHS_PER_FILE)]
    return [(f"{chip['chip_id']}_p{k}", block) for k, block in enumerate(blocks)]


def task_states():
    """state, compute used and error of every Earth Engine task, by task id."""
    out = {}
    for op in ee.data.listOperations():
        meta = op.get("metadata", {})
        out[op["name"].split("/")[-1]] = {"state": meta.get("state"),
                                          "eecu_s": meta.get("batchEecuUsageSeconds"),
                                          "error": op.get("error", {}).get("message")}
    return out


def export_file(chip, name, months):
    img, region = chip_image(chip, months)
    task = ee.batch.Export.image.toDrive(
        image=img, description=name, folder=DRIVE_FOLDER, fileNamePrefix=name,
        region=region, crs=chip["epsg"], crsTransform=[RES, 0, float(chip["x"]), 0, -RES, float(chip["y"])],
        maxPixels=1e9, fileFormat="GeoTIFF",
    )
    task.start()
    return {"file": name, "chip_id": chip["chip_id"], "task_id": task.id,
            "submitted": pd.Timestamp.now(tz="UTC").isoformat()}


log = pd.read_csv(EXPORT_LOG) if os.path.exists(EXPORT_LOG) else pd.DataFrame(columns=LOG_COLUMNS)
states = task_states()
ok_ids = {tid for tid, s in states.items() if s["state"] in OK_STATES}
ok_files = set(log.loc[log["task_id"].isin(ok_ids), "file"])

rows = manifest if ROWS is None else manifest.iloc[ROWS[0]:ROWS[1]]
todo = [(chip, name, months) for _, chip in rows.iterrows()
        for name, months in chip_files(chip) if name not in ok_files]
if MODE == "test":
    todo = todo[:1]
elif MODE != "all":
    todo = []

new = []
for chip, name, months in todo:
    new.append(export_file(chip, name, months))
    print("submitted", name)
log = pd.concat([log, pd.DataFrame(new, columns=LOG_COLUMNS)], ignore_index=True)
log.to_csv(EXPORT_LOG, index=False)
print(f"{len(new)} new exports, {len(ok_files)} already done or running in project {PROJECT_ID}")

## 6. How are the exports doing?

Rerun this cell any time. It also estimates the Earth Engine compute (EECU-hours) from the finished exports: our free tier gives 150 per month, and going over only makes things slower, it doesn't stop them.

In [ ]:
states = task_states()
log = pd.read_csv(EXPORT_LOG)
log["state"] = log["task_id"].map(lambda t: states.get(t, {}).get("state", "UNKNOWN"))
log["eecu_s"] = log["task_id"].map(lambda t: states.get(t, {}).get("eecu_s"))
latest = log.drop_duplicates("file", keep="last")
print(latest["state"].value_counts().to_dict(), "| files expected:", sum(len(chip_files(c)) for _, c in manifest.iterrows()))

done = latest[(latest["state"] == "SUCCEEDED") & latest["eecu_s"].notna()]
if len(done):
    per_file = done["eecu_s"].astype(float).mean() / 3600
    total = per_file * sum(len(chip_files(c)) for _, c in manifest.iterrows())
    print(f"~{per_file:.2f} EECU-hours per file -> ~{total:.0f} for everything")
for _, r in latest[latest["state"] == "FAILED"].iterrows():
    print("FAILED", r["file"], "->", states[r["task_id"]].get("error"))

## 7. Monthly weather per chip (gridMET)

One CSV for all chips: total rain (`pr`, mm), mean daily max/min temperature (`tmmx`, `tmmn`, Kelvin) and mean vapor pressure deficit (`vpd`, kPa), every month. gridMET pixels are 4 km, so this is one value per chip and month. Only runs once.

In [ ]:
GRIDMET = ee.ImageCollection("IDAHO_EPSCOR/GRIDMET")
WEATHER = "gridmet_monthly"


def gridmet_month(m):
    start = ee.Date(f"{m}-01")
    month = GRIDMET.filterDate(start, start.advance(1, "month"))
    return ee.Image.cat([month.select("pr").sum(), month.select(["tmmx", "tmmn", "vpd"]).mean()])


weather_done = log["file"].eq(WEATHER) & log["task_id"].isin(
    [tid for tid, s in task_states().items() if s["state"] in OK_STATES])
if MODE == "all" and not weather_done.any():
    chip_fc = ee.FeatureCollection([ee.Feature(chip_region(c["epsg"], c["x"], c["y"]), {"chip_id": str(c["chip_id"])})
                                    for _, c in manifest.iterrows()])
    first = min(manifest["start_month"])
    months = [str(p) for p in pd.period_range(first, END_MONTH, freq="M")]
    images = ee.ImageCollection([gridmet_month(m).set("month", m) for m in months])
    table = ee.FeatureCollection(images.map(
        lambda img: img.reduceRegions(chip_fc, ee.Reducer.mean(), scale=4000)
                       .map(lambda f: f.set("month", img.get("month")))).flatten())
    task = ee.batch.Export.table.toDrive(collection=table, description=WEATHER, folder=DRIVE_FOLDER,
                                         fileNamePrefix=WEATHER, fileFormat="CSV",
                                         selectors=["chip_id", "month", "pr", "tmmx", "tmmn", "vpd"])
    task.start()
    log = pd.concat([log[LOG_COLUMNS], pd.DataFrame([{"file": WEATHER, "chip_id": "", "task_id": task.id,
                                                      "submitted": pd.Timestamp.now(tz="UTC").isoformat()}])],
                    ignore_index=True)
    log.to_csv(EXPORT_LOG, index=False)
    print("submitted", WEATHER, f"({len(months)} months x {len(manifest)} chips)")
else:
    print("weather: already submitted" if weather_done.any() else "weather: set MODE = 'all' to export it")

## Next

- When every export is `SUCCEEDED`, the GeoTIFFs are in Drive → `cs497_hls_chips`, along with `gridmet_monthly.csv`.
- Step 3 copies them to Borah and turns them into training samples (NBR/NDVI, recovery %, splits by fire / year / region).
- Commit `data/chips_manifest.csv` and `figures/chips_per_fire.png`, with outputs cleared. The export log stays local.